# 01 · Introdução ao DuckDB  ⏱ 15 min

DuckDB é um banco de dados analítico (OLAP) que roda dentro do seu processo. Ele não tem servidor nem usuário e senha. É uma biblioteca, como o SQLite, só que otimizada para análise.

- Instala com `pip install duckdb` e roda no notebook, no script, no CLI e no navegador (WASM).
- Guarda dados por coluna e processa milhares de valores por vez, usando todos os núcleos da CPU.
- Lê CSV, Parquet, JSON, Excel, S3, HTTP, Postgres, MySQL, Iceberg e Delta direto, sem importar antes.
- Tem atalhos de SQL como `GROUP BY ALL`, `SELECT * EXCLUDE` e `COLUMNS(...)`.
- Processa dados maiores que a RAM, gravando resultados intermediários em disco quando precisa.

| | SQLite | Postgres | Pandas | Spark | **DuckDB** |
|---|---|---|---|---|---|
| Tipo de carga | transacional | transacional | análise | análise | **análise** |
| Servidor | não | sim | não | cluster | **não** |
| Escala típica | MB a GB | GB a TB | até a RAM | TB a PB | **GB a TB numa máquina** |

In [ ]:
import duckdb
duckdb.__version__

## Primeira consulta

O módulo `duckdb` já tem uma conexão padrão em memória.

In [ ]:
duckdb.sql("SELECT 'Olá, DuckDB!' AS mensagem, 21 * 2 AS resposta").show()

## Consultando um arquivo sem carregar nada

O nome do arquivo é a tabela. O DuckDB detecta o formato, o separador e os tipos sozinho.

In [ ]:
duckdb.sql("SELECT * FROM 'dados/clientes.csv' LIMIT 5").show()

In [ ]:
duckdb.sql("""
    SELECT canal, count(*) AS itens, round(sum(quantidade * preco_unitario) / 1e6, 2) AS receita_mi
    FROM 'dados/vendas.parquet'
    GROUP BY canal
    ORDER BY receita_mi DESC
""").show()

5 milhões de linhas agregadas em uma fração de segundo, sem `read_parquet` em memória antes.

## Consultando um DataFrame do Pandas

O DuckDB enxerga variáveis Python pelo nome (*replacement scan*) e evita copiar os dados quando consegue.

In [ ]:
import pandas as pd

metas = pd.DataFrame({
    "canal": ["site", "app", "marketplace", "loja_fisica"],
    "meta_mi": [710.0, 690.0, 720.0, 700.0],
})

duckdb.sql("""
    SELECT v.canal,
           round(sum(quantidade * preco_unitario) / 1e6, 2) AS receita_mi,
           m.meta_mi,
           receita_mi >= m.meta_mi AS bateu_meta
    FROM 'dados/vendas.parquet' v
    JOIN metas m USING (canal)
    GROUP BY v.canal, m.meta_mi
""").df()

## Conexões: em memória ou em arquivo

- `duckdb.connect()` cria um banco em memória (some ao fechar).
- `duckdb.connect("curso.duckdb")` cria um arquivo único, portátil, com todas as tabelas.

> **DuckDB 2.0:** arquivos criados pela 2.0 **não abrem** em versões 1.x. A 2.0 abre arquivos antigos normalmente.

In [ ]:
con = duckdb.connect("curso.duckdb")
con.sql("CREATE OR REPLACE TABLE clientes AS FROM 'dados/clientes.csv'")
con.sql("CREATE OR REPLACE TABLE produtos AS FROM 'dados/produtos.json'")
con.sql("CREATE OR REPLACE TABLE vendas   AS FROM 'dados/vendas.parquet'")
con.sql("SHOW TABLES").show()

In [ ]:
con.sql("DESCRIBE vendas").show()

## Do DuckDB para o Python e de volta

Todo resultado pode ser convertido para o formato que você usa: `.df()` (Pandas), `.pl()` (Polars), `.arrow()` (Arrow), `.fetchall()` (tuplas).

In [ ]:
top = con.sql("""
    SELECT p.categoria, round(sum(v.quantidade * v.preco_unitario), 2) AS receita
    FROM vendas v JOIN produtos p USING (produto_id)
    WHERE v.status = 'concluido'
    GROUP BY ALL ORDER BY receita DESC
""")
top.pl()

In [ ]:
ax = top.df().plot.barh(x="categoria", y="receita", legend=False, title="Receita por categoria")

## Quando **não** usar DuckDB

- Muitas escritas pequenas e concorrentes (app web transacional): use Postgres.
- Vários processos escrevendo no mesmo arquivo ao mesmo tempo: só um processo escreve por vez. (A 2.0 traz o servidor *Quack* para isso, veja o notebook 03.)
- Petabytes distribuídos: aí Spark/Trino/BigQuery fazem sentido.

In [ ]:
con.close()